# Z3-Python 05 — Quantificateurs et preuves formelles

**Navigation** : [Index](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [Serie Z3 C# (Z3.Linq)](../Z3/README.md) | [<< Z3-Python-04 Chaines](Z3-Python-04-Strings-Regex.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Utiliser** le quantificateur universel `ForAll` pour exprimer qu'une propriete hold pour toute valeur
2. **Utiliser** le quantificateur existentiel `Exists` pour exprimer l'existence d'au moins un temoin
3. **Prouver** qu'une formule est valide (un theoreme) via la technique de negation-et-verification
4. **Combiner** quantificateurs universels et existentiels dans des formules imbriquees
5. **Reconnaitre** les limites de Z3 : comprendre quand le solveur repond `unknown`

### Prerequis
- Z3-Python 01 (Introduction) : `Solver`, `Int`, `Real`, `Bool`, sat/unsat
- Notions de logique du premier ordre (quantificateurs $\forall$, $\exists$)

### Duree estimee : ~35 min

---

**Ce notebook** marque un tournant dans la serie : jusqu'ici (NB01-04), nous avons utilise Z3 pour trouver des **valeurs concretes** satisfaisant des contraintes (« existe-t-il un `x` tel que... ? »). Desormais, nous voulons **prouver des proprietes generales** : « pour tout `x`, cette egalite hold-t-elle ? » Cela requiere les **quantificateurs** `ForAll` ($\forall$) et `Exists` ($\exists$), et la technique de **preuve par refutation** : une formule est valide si et seulement si sa negation est insatisfiable.

> **Note technique** : Z3 est un solveur SMT, pas un assistant de preuve interactif comme Lean ou Coq. Une « preuve » ici signifie une **decision algorithmique** : le solveur confirme qu'aucun contre-exemple n'existe. Pour les fragments decidables (arithmetic lineaire, theories de base), cette preuve est **complete et automatique**. Pour les fragments plus riches (quantificateurs arbitraires sur les reels, nonlinear), Z3 peut repondre `unknown`.

In [1]:
# Imports et verification de l'environnement
from z3 import *

print(f"Imports OK : z3-solver version {get_version_string()}")

Imports OK : z3-solver version 4.16.0


## 1. Motivation : au-dela des valeurs concretes

Dans les notebooks precedents, nous avons pose des questions **existentielles concretes** :
- « Existe-t-il des entiers `x, y` tels que `x + y == 10` et `x > y` ? » -> Z3 repond `sat` et donne un modele.

Mais supposons que nous voulions **prouver une identite mathematique**. Par exemple, l'identite additive :
$$
\forall x \in \mathbb{R}, \quad x + 0 = x
$$

Nous pourrions tester quelques valeurs : `5 + 0 == 5` (vrai), `3.14 + 0 == 3.14` (vrai)... mais cela ne **prouve** rien : il y a une infinite de reels. Il nous faut un moyen d'exprimer « pour **tous** les `x` ».

C'est precisement le role des **quantificateurs** en logique du premier ordre :

| Quantificateur | Notation math | API Z3 | Sens |
|----------------|---------------|--------|------|
| Universel | $\forall x.\ F$ | `ForAll([x], F)` | F est vraie pour toute valeur de x |
| Existentiel | $\exists x.\ F$ | `Exists([x], F)` | Il existe au moins une valeur de x rendant F vraie |

Le principe central de ce notebook est la **technique de preuve par refutation** :

> Une formule $F$ est **valide** (un theoreme) si et seulement si sa negation $\neg F$ est **insatisfiable**.

| Negation de $F$ | Resultat de Z3 | Conclusion sur $F$ |
|-----------------|---------------|---------------------|
| `Not(F)` est `unsat` | Aucun contre-exemple | $F$ est **valide** (theoreme) |
| `Not(F)` est `sat` | Un contre-exemple existe | $F$ est **fausse** (contre-exemple donne) |
| `Not(F)` est `unknown` | Z3 ne peut pas conclure | Indecidable par ce solveur |

## 2. `ForAll` — le quantificateur universel

Le quantificateur `ForAll([x], F)` exprime que la formule `F` est vraie pour **toute** valeur de la variable `x`. La syntaxe prend une **liste** de variables en premier argument (on peut quantifier plusieurs variables a la fois).

Pour **prouver** qu'une formule universelle est valide, nous negons la formule entiere et verifions que cette negation est `unsat`. Si la negation est insatisfiable, alors la formule originale ne possede aucun contre-exemple : elle est donc un theoreme.

Prouvons l'identite additive $\forall x.\ x + 0 = x$ :

In [2]:
# Preuve de l'identite additive : pour tout x reel, x + 0 == x
x = Real('x')

# La formule que nous voulons prouver
formule = ForAll([x], x + 0 == x)
print("Formule a prouver :", formule)

# Technique : verifier que la NEGATION est insatisfiable
s = Solver()
s.add(Not(formule))  # Existe-t-il un x tel que x + 0 != x ?

resultat = s.check()
print("Negation de la formule :", resultat)

if resultat == unsat:
    print("=> Aucun contre-exemple trouve : la formule est VALIDE (theoreme).")
elif resultat == sat:
    print("=> Contre-exemple trouve :", s.model())
else:
    print("=> Z3 ne peut pas conclure (unknown).")

Formule a prouver : ForAll(x, x + 0 == x)
Negation de la formule : unsat
=> Aucun contre-exemple trouve : la formule est VALIDE (theoreme).


### Interpretation : preuve par refutation

**Sortie obtenue** : la negation `Not(ForAll([x], x + 0 == x))` est `unsat`, donc l'identite additive est valide.

| Etape | Operation | Resultat |
|-------|-----------|----------|
| 1 | Construire `ForAll([x], x + 0 == x)` | La formule a prouver |
| 2 | Ajouter `Not(formule)` au solveur | Cherche un contre-exemple |
| 3 | `s.check()` | `unsat` = aucun contre-exemple |
| 4 | Conclusion | La formule est un theoreme |

**Points cles** :
1. Z3 ne prouve pas directement « cette formule est valide ». Il prouve que sa negation est absurde.
2. Si la negation etait `sat`, `s.model()` donnerait un **contre-exemple** concret.
3. Le type `Real` couvre l'arithmetique rationnelle exacte ; l'identite hold sur tout $\mathbb{Q}$.

### 2.1 Trois exemples supplementaires

Prouvons trois proprietes elementaires de l'arithmetique reelle pour solidifier la technique. Pour chacune, nous negons la formule et verifions le resultat.

In [3]:
# Trois proprietes elementaires prouvees par refutation
x = Real('x')
y = Real('y')

proprietes = [
    ("Identite multiplicative : x * 1 == x", ForAll([x], x * 1 == x)),
    ("Commutativite de l'addition : x + y == y + x", ForAll([x, y], x + y == y + x)),
    ("Neutre additif a droite : 0 + x == x", ForAll([x], 0 + x == x)),
]

for nom, formule in proprietes:
    s = Solver()
    s.add(Not(formule))
    res = s.check()
    statut = "VALIDE" if res == unsat else ("FAUSSE" if res == sat else "UNKNOWN")
    print(f"{nom}")
    print(f"  => {statut} (negation = {res})")
    print()

Identite multiplicative : x * 1 == x
  => VALIDE (negation = unsat)

Commutativite de l'addition : x + y == y + x
  => VALIDE (negation = unsat)

Neutre additif a droite : 0 + x == x
  => VALIDE (negation = unsat)



### Interpretation : trois theoremes automatiques

**Sortie obtenue** : les trois proprietes sont validees comme `VALIDE`.

| Propriete | Formule Z3 | Verdict |
|-----------|-----------|---------|
| $x \times 1 = x$ | `ForAll([x], x * 1 == x)` | VALIDE |
| $x + y = y + x$ | `ForAll([x, y], x + y == y + x)` | VALIDE |
| $0 + x = x$ | `ForAll([x], 0 + x == x)` | VALIDE |

**Points cles** :
1. `ForAll` accepte une **liste** de variables : `ForAll([x, y], ...)` quantifie sur deux variables.
2. La technique est systematique : **negation -> check -> unsat signifie valide**.
3. Cette approche automatique contraste avec une preuve manuelle ou une demonstration par induction.

## 3. `Exists` — le quantificateur existentiel

Le quantificateur `Exists([x], F)` exprime qu'il existe **au moins une** valeur de `x` rendant `F` vraie. Contrairement a `ForAll`, on peut directement chercher un **temoin** concret.

Notons le lien logique fondamental : la negation d'un existentiel est un universel.
$$
\neg \exists x.\ F \quad \equiv \quad \forall x.\ \neg F
$$

Examinons deux cas : l'un satisfiable, l'autre insatisfiable.

In [4]:
# Exemple satisfiable : existe-t-il un x reel tel que x*x == 4 ?
x = Real('x')

# Etape 1 : prouver l'EXISTENCE avec le quantificateur existentiel.
formule = Exists([x], x * x == 4)
s = Solver()
s.add(formule)
res = s.check()
print("Formule   :", formule)
print("Existence :", res, "(un temoin existe)")

# Piege classique : la variable x est LIEE par Exists. Elle n'apparait donc
# PAS dans le modele de la formule quantifiee -> m[x] vaut None, jamais un temoin.
print("m[x] (x lie par Exists) :", s.model()[x], " <- aucun temoin lisible ici")

# Etape 2 : pour OBTENIR un temoin concret, on skolemise : on reasserte la meme
# contrainte sur une constante LIBRE, dont la valeur est lisible dans le modele.
racine = Real('racine')
s2 = Solver()
s2.add(racine * racine == 4)
s2.check()
temoin = s2.model()[racine]
print(f"Temoin concret (constante libre) : racine = {temoin}")
print(f"Verification : {temoin} * {temoin} =", simplify(temoin * temoin), "(attendu : 4)")

Formule   : Exists(x, x*x == 4)
Existence : sat (un temoin existe)
m[x] (x lie par Exists) : None  <- aucun temoin lisible ici
Temoin concret (constante libre) : racine = 2
Verification : 2 * 2 = 4 (attendu : 4)


### Interpretation : prouver l'existence puis extraire le temoin

**Sortie obtenue** : `Exists([x], x * x == 4)` est `sat` — un temoin existe. Mais `m[x]` vaut `None` : la variable `x` est **liee** par le quantificateur, elle n'apparait donc pas dans le modele. Pour lire un temoin concret, on **skolemise** : on reasserte la contrainte sur une constante **libre** (`racine`), et `m[racine]` donne alors une racine concrete (ici `2`, Z3 pouvant renvoyer l'une des deux racines `2` ou `-2`).

| Etape | Operation | Resultat |
|-------|-----------|----------|
| 1 | `Exists([x], x*x == 4)` puis `check()` | `sat` — l'existence est prouvee |
| 2 | `m[x]` sur la variable liee | `None` — aucun temoin lisible |
| 3 | Reasserter `racine*racine == 4` (constante libre) | `m[racine] = 2` — temoin concret |

**Points cles** :
1. Une variable **liee** par `Exists`/`ForAll` n'apparait **pas** dans `s.model()` : `m[x]` y vaut `None`. Le quantificateur prouve l'existence, il ne livre pas le temoin.
2. Pour **obtenir** le temoin, on skolemise : la meme contrainte posee sur une **constante libre** rend sa valeur lisible via `m[racine]`.
3. C'est la difference fondamentale entre *prouver qu'un temoin existe* (`Exists` + `sat`) et *exhiber ce temoin* (contrainte sur une constante libre).

In [5]:
# Exemple insatisfiable : existe-t-il un x reel tel que x*x < 0 ?
x = Real('x')

formule = Exists([x], x * x < 0)
print("Formule :", formule)
print("(Un carre reel est toujours positif ou nul)")

s = Solver()
s.add(formule)
res = s.check()
print("Resultat :", res)
if res == unsat:
    print("=> Aucun reel x tel que x*x < 0 : la formule est FAUSSE.")
elif res == sat:
    print("=> Temoin trouve :", s.model())

Formule : Exists(x, x*x < 0)
(Un carre reel est toujours positif ou nul)
Resultat : unsat
=> Aucun reel x tel que x*x < 0 : la formule est FAUSSE.


### Interpretation : existentiel insatisfiable

**Sortie obtenue** : la formule `Exists([x], x * x < 0)` est `unsat`, ce qui prouve qu'aucun carre reel n'est strictement negatif.

| Contraste | `Exists` sat | `Exists` unsat |
|-----------|-------------|----------------|
| Sens | Un temoin existe | Aucun temoin possible |
| Exemple | `Exists([x], x*x == 4)` | `Exists([x], x*x < 0)` |
| Conclusion | La propriete est realisable | La propriete est impossible |

**Points cles** :
1. `Exists` directement satisfiable (pas besoin de negation) : le solveur cherche un temoin.
2. `unsat` sur `Exists` signifie que la propriete n'est **jamais** realisable.
3. Le domaine compte : sur les `Real`, un carre est toujours positif ; sur les `Int` aussi.

## 4. Preuves formelles

Maintenant que nous maitrisons `ForAll`, `Exists` et la technique de refutation, nous pouvons prouver des **theoremes** plus substentiels. La demarche est toujours la meme :

1. **Enoncer** la propriete sous forme de formule `ForAll`.
2. **Nier** la formule entiere.
3. **Verifier** que la negation est `unsat`.

### Qu'est-ce qu'une « preuve » en Z3 ?

Z3 est un **solveur de decision** : pour les fragments decidables (arithmetic lineaire sur $\mathbb{Z}$ et $\mathbb{R}$, theories de base), il repond de maniere **certaine** — `sat` ou `unsat`. Une « preuve » Z3 signifie donc : *le solveur a confirme qu'aucun contre-exemple n'existe*. Ce n'est pas une preuve constructrice comme dans Lean ou Coq (ou l'on construit un terme de preuve pas-a-pas), mais une **verification algorithmique complete**.

Pour des fragments **indecidables** (quantificateurs arbitraires sur les reels, nonlinear), Z3 peut repondre `unknown` (cf. Section 5).

Prouvons deux proprietes : la trichotomie et la monotonicite de la fonction carre sur les positifs.

In [6]:
# Preuve 1 : trichotomie sur les reels
# Pour tout x reel : x < 0 OU x == 0 OU x > 0 (un des trois cas)
x = Real('x')

trichotomie = ForAll([x], Or(x < 0, x == 0, x > 0))
print("Theoreme de trichotomie :", trichotomie)

s = Solver()
s.add(Not(trichotomie))
res = s.check()
print(f"Negation = {res}")
print(f"=> Trichotomie : {'VALIDE' if res == unsat else 'NON PROUVEE'}")

print()

# Preuve 2 : monotonicite du carre sur les reels positifs
# Pour tous x, y >= 0 : si x <= y alors x*x <= y*y
y = Real('y')

monotonicite = ForAll([x, y], Implies(
    And(x >= 0, y >= 0, x <= y),
    x * x <= y * y
))
print("Monotonicite du carre (x >= 0) :", monotonicite)

s2 = Solver()
s2.add(Not(monotonicite))
res2 = s2.check()
print(f"Negation = {res2}")
print(f"=> Monotonicite : {'VALIDE' if res2 == unsat else 'NON PROUVEE'}")

Theoreme de trichotomie : ForAll(x, Or(x < 0, x == 0, x > 0))
Negation = unsat
=> Trichotomie : VALIDE

Monotonicite du carre (x >= 0) : ForAll([x, y],
       Implies(And(x >= 0, y >= 0, x <= y), x*x <= y*y))
Negation = unsat
=> Monotonicite : VALIDE


### Interpretation : theoremes prouves automatiquement

**Sortie obtenue** : les deux proprietes sont validees (`unsat` sur la negation).

| Theoreme | Formule | Verdict |
|----------|---------|---------|
| Trichotomie | $\forall x.\ x < 0 \lor x = 0 \lor x > 0$ | VALIDE |
| Monotonicite | $\forall x, y \geq 0.\ x \leq y \Rightarrow x^2 \leq y^2$ | VALIDE |

**Points cles** :
1. `Implies(p, q)` correspond a l'implication logique $p \Rightarrow q$.
2. `And(...)` permet de combiner plusieurs hypotheses dans l'antecedent de l'implication.
3. Z3 traite l'arithmetic lineaire reelle comme un fragment **decidable** : la preuve est complete et certaine.
4. Le theoreme de monotonicite utilise l'arithmetic **non lineaire** ($x^2$, $y^2$) ; Z3 parvient tout de meme a le decider.

## Exercice 1 : Prouver l'identite additive

### Enonce

Ecrivez une fonction `prouver_identite_additive()` qui prouve que `ForAll([x], x + 0 == x)` est valide (pour `x` reel) en utilisant la technique de refutation : verifier que la negation est `unsat`.

La fonction doit retourner `True` si la formule est valide, `False` sinon.

### Indices

- `# Indice` : utilisez `Not(ForAll([x], x + 0 == x))` comme contrainte.
- `# Etape 1` : declarer `x = Real('x')`.
- `# Etape 2` : creer un `Solver()`, ajouter la negation.
- `# Etape 3` : si `s.check() == unsat`, la formule est valide -> retourner `True`.

In [7]:
# EXERCICE 1 : Prouver l'identite additive par refutation.

def prouver_identite_additive() -> bool:
    """Prouve que ForAll([x], x + 0 == x) est valide sur les reels.

    Retourne True si valide, False sinon.

    # Indice : utilise Not(ForAll(...)) puis s.check().
    # Etape 1 : declarer x = Real('x')
    # Etape 2 : creer un Solver, ajouter Not(ForAll([x], x + 0 == x))
    # Etape 3 : si s.check() == unsat -> return True
    """
    print("Exercice 1 - a completer")
    # TODO etudiant : implémentez la preuve par refutation
    return None  # TODO etudiant : remplacer par True ou False

valide = prouver_identite_additive()
print("Identite additive valide :", valide)

Exercice 1 - a completer
Identite additive valide : None


## 5. Quantificateurs combines et limites

Les quantificateurs peuvent etre **imbriques** : une formule peut contenir un `ForAll` et un `Exists` en meme temps. Par exemple, l'enonce « pour tout reel `x`, il existe un reel `y` strictement plus grand que `x` » s'ecrit :
$$
\forall x.\ \exists y.\ y > x
$$
Cette propriete exprime qu'il n'y a pas de « plus grand reel » : on peut toujours trouver un nombre plus grand.

### La limite de Z3 : `unknown`

Z3 est puissant, mais **indecidable** sur certains fragments. Lorsque le solveur ne peut pas conclure, il repond `unknown`. Cela arrive typiquement avec :
- De l'arithmetic **non lineaire entiere** (multiplication de variables sur $\mathbb{Z}$) — indecidable en general (10e probleme de Hilbert)
- De l'arithmetic non lineaire reelle non bornee
- Des quantificateurs imbriques complexes
- Des formules hors des fragments decides

Il est important de comprendre que `unknown` ne signifie ni `sat` ni `unsat` : le solveur **abandonne**, sans conclusion. Ce n'est pas une erreur, c'est une limite fondamentale de la decision automatique. Pour eviter une recherche sans fin, on borne le solveur avec un **timeout** : au-dela du budget, Z3 renvoie `unknown` plutot que de boucler.

In [8]:
# Quantificateurs imbriques : pour tout x, il existe y > x (pas de plus grand reel)
x = Real('x')
y = Real('y')

pas_de_plus_grand = ForAll([x], Exists([y], y > x))
print("Formule :", pas_de_plus_grand)

s = Solver()
s.add(Not(pas_de_plus_grand))
res = s.check()
print(f"Negation = {res}")
if res == unsat:
    print("=> VALIDE : il n'existe pas de plus grand reel (theoreme).")
elif res == sat:
    print("=> FAUSSE : contre-exemple trouve.", s.model())
else:
    print("=> Z3 ne peut pas conclure (unknown).")

Formule : ForAll(x, Exists(y, y > x))
Negation = unsat
=> VALIDE : il n'existe pas de plus grand reel (theoreme).


### Interpretation : quantificateurs imbriques

**Sortie obtenue** : la negation de « pour tout `x`, il existe `y > x` » est `unsat`, donc la propriete est valide.

| Aspect | Detail |
|--------|--------|
| Formule | `ForAll([x], Exists([y], y > x))` |
| Sens | Pour tout reel, il en existe un plus grand |
| Negation | `Not(ForAll([x], Exists([y], y > x)))` |
| Verdict | `unsat` -> formule valide |

**Points cles** :
1. `ForAll` et `Exists` se combinent naturellement dans une meme formule.
2. La negation d'une formule imbriquee negatione l'**ensemble** : `Not(ForAll([x], Exists([y], ...)))`.
3. L'ordre des quantificateurs compte : $\forall x.\ \exists y$ n'est pas equivalent a $\exists y.\ \forall x$.

In [9]:
# Cas ou Z3 repond REELLEMENT unknown : arithmetic non lineaire entiere.
# Existe-t-il des entiers a, b, c > 1 tels que a^3 + b^3 == c^3 ?
# (Le dernier theoreme de Fermat dit non, mais cette preuve depasse Z3 :
#  l'arithmetic non lineaire entiere est indecidable en general.)
# Sans budget de temps, Z3 chercherait indefiniment -> on borne avec un timeout.
a, b, c = Ints('a b c')

s = Solver()
s.set("timeout", 3000)  # 3 secondes ; au-dela, Z3 abandonne
s.add(a > 1, b > 1, c > 1, a*a*a + b*b*b == c*c*c)

print("Probleme : a^3 + b^3 == c^3  avec a, b, c > 1")
res = s.check()
print(f"Resultat : {res}")

if res == sat:
    print("=> Temoin trouve :", s.model())
elif res == unsat:
    print("=> Aucune solution (prouve).")
else:
    print(f"=> UNKNOWN : Z3 abandonne (raison : {s.reason_unknown()}).")
    print("   L'arithmetic non lineaire entiere est indecidable en general :")
    print("   ce n'est PAS un bug, mais une limite fondamentale de la decision automatique.")

Probleme : a^3 + b^3 == c^3  avec a, b, c > 1


Resultat : unknown
=> UNKNOWN : Z3 abandonne (raison : timeout).
   L'arithmetic non lineaire entiere est indecidable en general :
   ce n'est PAS un bug, mais une limite fondamentale de la decision automatique.


### Interpretation : honnetete face a `unknown`

**Sortie obtenue** : sur `a^3 + b^3 == c^3` (avec `a, b, c > 1`), Z3 ne tranche pas dans le budget imparti et repond `unknown` (raison : `timeout`). Le probleme — un cas du dernier theoreme de Fermat — est hors de portee de la decision automatique : l'arithmetic non lineaire entiere est indecidable en general. Le `timeout` borne la recherche ; sans lui, `check()` ne terminerait pas.

| Theorie | Resultat typique | Interpretation |
|---------|-----------------|----------------|
| Arithmetic lineaire reelle | `sat` ou `unsat` | Toujours decidable |
| Arithmetic lineaire entiere | `sat` ou `unsat` | Decidable (Presburger) |
| Non lineaire reelle (bornee) | `sat`/`unsat` ou `unknown` | Parfois decidable |
| Non lineaire entiere | souvent `unknown` | Indecidable en general |

> **Note technique** : `unknown` n'est pas un echec du a un bug. C'est une **limite theorique** : par l'indecidabilite de l'arithmetic entiere du premier ordre (theoreme de Matiiassevitch / 10e probleme de Hilbert), aucun algorithme ne peut decider toutes les formules. Z3 fait de son mieux avec des heuristiques puissantes, mais certaines formules restent hors de portee. Quand cela arrive, on peut : (a) simplifier la formule, (b) limiter le domaine (passer a un intervalle entier borne), (c) utiliser une tactique specialisee, (d) accepter `unknown` comme reponse honnete.

## Exercice 2 : Verifier l'inexistence d'un carre negatif

### Enonce

Ecrivez une fonction `existe_carre_negatif()` qui verifie si `Exists([x], x * x < 0)` est satisfiable sur les reels. Retournez `True` si sat (un carre negatif existe), `False` si unsat (aucun carre negatif).

### Indices

- `# Indice` : attention au type. `Real('x')` donne des reels ; un carre reel est toujours positif.
- `# Etape 1` : declarer `x = Real('x')`.
- `# Etape 2` : creer un `Solver`, ajouter `Exists([x], x * x < 0)`.
- `# Etape 3` : `s.check() == sat` -> retourner `True`, sinon `False`.

In [10]:
# EXERCICE 2 : Verifier si un carre negatif existe sur les reels.

def existe_carre_negatif() -> bool:
    """Verifie si Exists([x], x*x < 0) est sat sur les reels.

    Retourne True si satisfiable, False sinon.

    # Indice : utilisez Real('x') (un carre reel est toujours >= 0).
    # Etape 1 : declarer x = Real('x')
    # Etape 2 : creer un Solver, ajouter Exists([x], x * x < 0)
    # Etape 3 : si s.check() == sat -> return True, sinon False
    """
    print("Exercice 2 - a completer")
    # TODO etudiant : implémentez la vérification
    return None  # TODO etudiant : remplacer par True ou False

resultat = existe_carre_negatif()
print("Un carre negatif existe :", resultat)

Exercice 2 - a completer
Un carre negatif existe : None


## 6. Model checking avec quantificateurs

Le **model checking** (verification de modele) consiste a trouver une assignation concrete satisfaisant un ensemble de contraintes — y compris des contraintes quantifiees. Contrairement a la preuve (ou l'on cherche `unsat` sur une negation), le model checking cherche un **modele** (`sat` + `s.model()`).

Un exemple classique : existe-t-il un reel `x` tel que, pour tout reel `y` positif, `x <= y` ? En d'autres termes, existe-t-il un reel minimal par rapport aux positifs ?
$$
\exists x.\ \forall y.\ (y \geq 0 \Rightarrow x \leq y)
$$
Sur les reels, cette formule est satisfiable si `x <= 0` (n'importe quel `x` negatif ou nul est inferieur a tout `y >= 0`).

**Pour lire le temoin** `x`, on applique la meme technique qu'a la Section 3 : on **skolemise** le `Exists x` externe en declarant `x` comme une constante **libre**, de sorte que le `ForAll` ne porte plus que sur `y`. La valeur de `x` devient alors directement lisible dans le modele via `m[x]` (au lieu de rester un symbole non evalue si `x` etait lie par `Exists`). Trouvons un tel modele.

In [11]:
# Model checking : existe-t-il x tel que pour tout y >= 0, x <= y ?
# x est le temoin recherche (le "Exists x" externe). Pour lire sa valeur dans
# le modele, on le SKOLEMISE : on le declare comme constante LIBRE, et le ForAll
# ne porte plus que sur y. Sa valeur devient alors lisible via m[x].
x = Real('x')
y = Real('y')

contrainte = ForAll([y], Implies(y >= 0, x <= y))
print("Contrainte sur x (temoin libre) :", contrainte)

s = Solver()
s.add(contrainte)
res = s.check()
print(f"Resultat : {res}")

if res == sat:
    m = s.model()
    val_x = m[x]   # x est libre -> sa valeur est lisible dans le modele
    print(f"Modele trouve : x = {val_x}")
    print(f"Interpretation : {val_x} est <= a tout y >= 0 (un minorant des reels positifs)")
    print(f"Verification : {val_x} <= 1/10 ?", is_true(simplify(val_x <= RealVal('1/10'))))
    print(f"Verification : {val_x} <= 1/1000 ?", is_true(simplify(val_x <= RealVal('1/1000'))))
elif res == unsat:
    print("=> Aucun modele : la formule est fausse.")
else:
    print("=> Z3 ne peut pas conclure (unknown).")

Contrainte sur x (temoin libre) : ForAll(y, Implies(y >= 0, x <= y))
Resultat : sat
Modele trouve : x = 0
Interpretation : 0 est <= a tout y >= 0 (un minorant des reels positifs)
Verification : 0 <= 1/10 ? True
Verification : 0 <= 1/1000 ? True


### Interpretation : extraction du temoin par skolemisation

**Sortie obtenue** : Z3 trouve un modele ou `x = 0` — un reel inferieur ou egal a tout `y >= 0`. La valeur exacte depend du solveur, mais elle est bien **lisible** ici parce que `x` a ete declare comme constante **libre**.

| Aspect | Valeur |
|--------|--------|
| Question posee | $\exists x.\ \forall y.\ (y \geq 0 \Rightarrow x \leq y)$ |
| Formule resolue (skolemisee) | `ForAll([y], Implies(y >= 0, x <= y))`, `x` libre |
| Resultat | `sat` |
| Modele | `x = 0` (un minorant des reels positifs) |

**Points cles** :
1. Le `Exists x` externe est **skolemise** : on transforme « il existe `x` » en « voici la constante `x` » (variable libre). Le `ForAll` ne porte plus que sur `y`.
2. Une variable **liee** par un quantificateur ne peut **pas** etre extraite du modele : `m[x]` y vaut `None` et `m.evaluate(x)` renvoie le symbole `x` non evalue. Skolemiser (constante libre) est la facon standard de rendre le temoin lisible — c'est exactement la technique de la Section 3.
3. `Implies(y >= 0, x <= y)` restreint la contrainte universelle aux `y` positifs uniquement.
4. Le contraste avec la Section 4 : la, on cherchait `unsat` (preuve) ; ici, on cherche `sat` (modele) et on **lit** le temoin.

## Exercice 3 : Prouver l'absence de plus grand reel

### Enonce

Ecrivez une fonction `prouver_pas_de_plus_grand()` qui prouve que `ForAll([x], Exists([y], y > x))` est valide sur les reels (pour tout reel `x`, il existe un reel `y` strictement plus grand). Utilisez la technique de refutation.

Retournez `True` si la formule est valide (negation `unsat`), `False` sinon.

### Indices

- `# Indice` : niez la **formule entiere** : `Not(ForAll([x], Exists([y], y > x)))`.
- `# Etape 1` : declarer `x = Real('x')` et `y = Real('y')`.
- `# Etape 2` : creer un `Solver`, ajouter la negation.
- `# Etape 3` : si `s.check() == unsat`, la formule est valide -> retourner `True`.

In [12]:
# EXERCICE 3 : Prouver qu'il n'existe pas de plus grand reel.

def prouver_pas_de_plus_grand() -> bool:
    """Prouve que ForAll([x], Exists([y], y > x)) est valide sur les reels.

    Retourne True si valide, False sinon.

    # Indice : niez toute la formule avec Not(ForAll([x], Exists([y], y > x))).
    # Etape 1 : declarer x = Real('x'), y = Real('y')
    # Etape 2 : creer un Solver, ajouter Not(ForAll([x], Exists([y], y > x)))
    # Etape 3 : si s.check() == unsat -> return True
    """
    print("Exercice 3 - a completer")
    # TODO etudiant : implémentez la preuve par refutation
    return None  # TODO etudiant : remplacer par True ou False

valide = prouver_pas_de_plus_grand()
print("Pas de plus grand reel (valide) :", valide)

Exercice 3 - a completer
Pas de plus grand reel (valide) : None


## Recapitulatif

Ce notebook a introduit les **quantificateurs** et la **preuve formelle** avec Z3 :

| Concept | API Z3 | Usage |
|---------|--------|-------|
| Quantificateur universel | `ForAll([x], F)` | Prouver qu'une propriete hold pour toute valeur |
| Quantificateur existentiel | `Exists([x], F)` | Trouver un temoin verifiant une propriete |
| Preuve par refutation | `Solver().add(Not(F))` | F est valide ssi `Not(F)` est `unsat` |
| Quantificateurs imbriques | `ForAll([x], Exists([y], ...))` | Exprimer des proprietes relationnelles |
| Model checking | `sat` + `s.model()` | Extraire un modele concret |
| Limite de decision | `unknown` | Z3 ne peut pas conclure (indecidable) |

**Points essentiels a retenir** :
1. **`ForAll` vs `Exists`** : le premier exige une propriete universelle, le second cherche un temoin.
2. **Validite = negation insatisfiable** : pour prouver qu'une formule est un theoreme, on verifie que sa negation est `unsat`. Aucun contre-exemple n'existe.
3. **`unknown` est honnete** : Z3 est indecidable sur certains fragments (non lineaire, quantificateurs complexes). `unknown` n'est ni `sat` ni `unsat`, c'est un abandon.
4. **Contraste avec NB01** : dans le notebook d'introduction, nous cherchions des valeurs concretes (`sat` + modele). Desormais, nous prouvons des proprietes **generales** (`unsat` sur la negation = theoreme).
5. **Preuve Z3 vs preuve Lean** : Z3 est une **decision automatique** (algorithme), pas une preuve interactive (construction pas-a-pas). Les deux approches sont complementaires.

Le notebook suivant explorera l'**optimisation avancee** : objectifs multiples, front de Pareto, et `Optimize` hierarchique.